figs1  
四张图拼接 

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import myfunction as mf
from path_config import *
import seaborn as sns
import matplotlib.pyplot as plt


把变量分组，然后分别展示前5

In [ ]:
def draw_rfecv_top(df_data, save_path, str_remark, top_num=5, show=True):
    df_o = df_data.copy()
    if str_remark == 'swnl':
        x_str = 'Importance'
        y_str = 'full_name'
        color = '#84aeb8'
    elif str_remark == 'swl':
        x_str = 'R2'
        y_str = 'full_name'
        color = '#84aeb8'
    elif str_remark == 'lrnl':
        x_str = 'Importance'
        y_str = 'full_name'
        color = '#7d84a8'
    elif str_remark == 'lrl':
        x_str = 'R2'
        y_str = 'full_name'
        color = '#7d84a8'
    df_o = df_o.sort_values(by=x_str, ascending=False).head(top_num)
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, top_num))
    sns.barplot(x=x_str, y=y_str, data=df_o, ax=ax, color=color,width=0.4)
    sns.despine(ax=ax, offset=10, trim=True, right=True, bottom=True, top=False)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_xlabel('', fontsize=16, labelpad=10)
    ax.set_ylabel('', fontsize=16, labelpad=10)
    ax.set_title(x_str, fontsize=15)
    for p in ax.patches:
        width = p.get_width()
        ax.text(width, p.get_y() + p.get_height()/2, '{:.3f}'.format(width), ha = 'left', va = 'center')
    plt.subplots_adjust(left=0.55,right=0.9,bottom=0.15,top=0.7)
    plt.savefig(save_path, dpi = 300)
    plt.tight_layout()
    if show:
        plt.show()
    else:
        plt.close()


def draw_type_top(df_mlr, df_ml, df_meta, str_marker, top_num=5):

    df_meta_filtered = df_meta[['var_name', 'full_name', 'var_type3']]


    df_mlr_merged = df_mlr.merge(
        df_meta_filtered,
        left_on='variables',
        right_on='var_name',
        how='left'
    )


    df_ml_merged = df_ml.merge(
        df_meta_filtered,
        left_on='Feature',
        right_on='var_name',
        how='left'
    )
    df_ml_merged = df_ml_merged[df_ml_merged['Rank'] ==1]


    if 'year' in df_ml_merged['var_name'].values:
        df_ml_merged = df_ml_merged[df_ml_merged['var_name'] != 'year']
    if 'year' in df_mlr_merged['var_name'].values:
        df_mlr_merged = df_mlr_merged[df_mlr_merged['var_name'] != 'year']

    save_path_ml = path_part1_fig + "figs1_bar_"+ str_marker +"_rfecv.svg"
    save_path_mlr = path_part1_fig + "figs1_bar_"+ str_marker +"_mlr.svg"
    draw_rfecv_top(df_ml_merged, save_path_ml, str_marker+'nl', top_num=top_num, show=False)
    draw_rfecv_top(df_mlr_merged, save_path_mlr, str_marker+'l', top_num=top_num, show=False)


    print("df_ml_merged var_type3 counts:")
    print(df_ml_merged['var_type3'].value_counts())
    print("df_sw_mlr_merged var_type3 counts:")
    print(df_mlr_merged['var_type3'].value_counts())


    var_type3_counts_sw = df_ml_merged['var_type3'].value_counts().to_dict()
    var_type3_counts_mlr = df_mlr_merged['var_type3'].value_counts().to_dict()


    for var_type, count in var_type3_counts_sw.items():
        if count > top_num:
            top_num = top_num
        else:
            top_num = count # 1
        df_ml_merged_filtered = df_ml_merged[df_ml_merged['var_type3'] == var_type]
        save_path_ml = path_part1_fig + "figs1_bar_"+str_marker+"_rfecv_" + str(var_type) + ".svg"
        draw_rfecv_top(df_ml_merged_filtered, save_path_ml, str_marker+'nl', top_num=top_num, show=False)
    for var_type, count in var_type3_counts_mlr.items():
        if count > top_num:
            top_num = top_num
        else:
            top_num = count # 1
        df_mlr_merged_filtered = df_mlr_merged[df_mlr_merged['var_type3'] == var_type]
        save_path_mlr = path_part1_fig + "figs1_bar_"+str_marker+"_mlr_" + str(var_type) + ".svg"
        draw_rfecv_top(df_mlr_merged_filtered, save_path_mlr, str_marker+'l', top_num=top_num, show=False)

In [11]:
df_sw_mlr = pd.read_csv(path_part2_pre + 'mlr_sw.csv')
df_sw_rf = pd.read_csv(path_part3_sw + 'sw_rfecv_features_RFcv.csv')

df_lr_mlr = pd.read_csv(path_part2_pre + 'mlr_lr_sw.csv')
df_lr_rf = pd.read_csv(path_part3_lrsw + 'lr_sw_rfecv_features_LGBMcv.csv')

df_meta = pd.read_csv(path_file + meta_file)

draw_type_top(df_sw_mlr, df_sw_rf, df_meta, 'sw', 3)
draw_type_top(df_lr_mlr, df_lr_rf, df_meta, 'lr', 3)

df_ml_merged var_type3 counts:
1    22
2    10
0     8
Name: var_type3, dtype: int64
df_sw_mlr_merged var_type3 counts:
1    9
2    6
0    4
Name: var_type3, dtype: int64
df_ml_merged var_type3 counts:
1    24
0    11
2    11
3     5
Name: var_type3, dtype: int64
df_sw_mlr_merged var_type3 counts:
2    11
1    11
0    10
3     1
Name: var_type3, dtype: int64


C:\Users\dell\AppData\Local\Temp\ipykernel_49932\4006515792.py:32: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
